# 02 - Data Preprocessing

## Objective

Prepare the Telco Customer Churn dataset for machine learning by:
- defining the target variable
- separating features and target
- splitting data into train and test sets
- encoding categorical variables
- scaling numerical variables
- avoiding data leakage

In [1]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Preprocessing Strategy

The preprocessing steps will follow a leakage-safe workflow:

1. Drop non-predictive identifier columns.
2. Convert `TotalCharges` from string to numeric and handle hidden missing values.
3. Encode the target variable `Churn` as binary values.
4. Separate features (`X`) and target (`y`).
5. Split the data into training and test sets.
6. Fit encoders and scalers only on the training data.
7. Transform both training and test data using the fitted preprocessing objects.

This avoids data leakage because the test set remains unseen during preprocessing.

In [2]:
import pandas as pd
# Drop customerID only if it exists
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = df.drop(columns=["customerID"])

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Count missing values after conversion
missing_after_conversion = df["TotalCharges"].isna().sum()

print(missing_after_conversion)

11


After converting `TotalCharges` to numeric, 11 missing values appeared.

These were hidden whitespace values in the original dataset. They were not detected by `df.info()` because whitespace is still treated as a string, not as a missing value.

From the EDA, these rows correspond to customers with `tenure = 0`, meaning they have not been billed yet. Therefore, imputing `TotalCharges` with 0 is business-logical.

The 11 missing `TotalCharges` values correspond to customers with `tenure = 0`, meaning they have just started their contract and have not yet accumulated any charges.

Dropping these rows would remove valid customers from the dataset, and mean imputation would incorrectly assign historical charges to customers who have not been billed yet.

Therefore, `TotalCharges` is imputed with 0, which is consistent with the business meaning of the feature.

In [3]:
# Fill missing TotalCharges values with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Confirm there are no missing values left
df["TotalCharges"].isna().sum()

np.int64(0)

Mean imputation is not appropriate here because the missing values are not random. They occur for customers with `tenure = 0`, meaning they have not yet accumulated any charges.

Assigning the average `TotalCharges` would incorrectly make new customers look like they have historical billing activity. Since `TotalCharges = 0` matches the business meaning, imputing with 0 is the most appropriate choice.

## Target Encoding

The target variable `Churn` is encoded as:

- `No` → 0
- `Yes` → 1

Churn is treated as the positive class because the business goal is to identify customers who are likely to leave. This makes recall directly interpretable as the percentage of actual churners correctly detected by the model.

In [4]:
# Encode target variable
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Check target distribution
df["Churn"].value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

The target distribution shows that approximately 26.5% of customers churned, while 73.5% stayed.

This confirms a moderate class imbalance. Since the imbalance is not extreme, the first modeling approach will use `class_weight='balanced'` instead of applying SMOTE immediately.

## Train/Test Split Strategy

The dataset is split into training and test sets using stratification.

`stratify=y` ensures that the churn/non-churn ratio is preserved in both sets. This is important because the dataset has a moderate class imbalance, and the model must be evaluated on a test set that fairly represents both churned and non-churned customers.

If the test set contains too few churn customers, evaluation metrics such as recall become unreliable because they are calculated on too few positive cases.

This could make the model appear stronger than it really is, especially if accuracy is high due to the majority non-churn class. Stratification helps ensure the test set fairly represents the original churn distribution.

## Feature and Target Separation

The dataset is separated into:

- `X`: feature variables used to predict churn
- `y`: target variable representing whether the customer churned

The target variable `Churn` has already been encoded as binary values, where `1` represents churn and `0` represents no churn.

In [5]:
# Separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


## Train/Test Split

The data is split into training and test sets before fitting any preprocessing objects.

This ensures that encoders and scalers are fitted only on the training data, preventing information from the test set from leaking into the training process.

In [6]:
from sklearn.model_selection import train_test_split

# Split features and target into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5634, 19)
X_test shape: (1409, 19)
y_train shape: (5634,)
y_test shape: (1409,)


In [7]:
print("Full dataset churn distribution:")
print(y.value_counts(normalize=True))

print("\nTraining set churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest set churn distribution:")
print(y_test.value_counts(normalize=True))

Full dataset churn distribution:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Training set churn distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Test set churn distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


The churn distribution is nearly identical across the full dataset, training set, and test set.

This confirms that `stratify=y` worked correctly and preserved the class balance during the split. As a result, the test set provides a fair evaluation sample for measuring recall on churn customers.

## Feature Type Classification

Features are separated based on how they should be preprocessed:

- Numerical features: continuous values with meaningful magnitude, such as tenure and charges.
- Binary features: already encoded 0/1 indicators, such as `SeniorCitizen`.
- Categorical features: text-based groups or service types that require encoding.

`SeniorCitizen` is kept separate because it is already encoded as a binary indicator. Scaling it would reduce interpretability without adding useful information.

In [8]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

binary_features = ["SeniorCitizen"]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [9]:
print("Numeric features:", len(numeric_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(binary_features) + len(categorical_features))

Numeric features: 3
Binary features: 1
Categorical features: 15
Total features: 19


## Categorical Encoding Strategy

Categorical variables are encoded using one-hot encoding.

Manual numeric mapping is avoided for multi-class categorical features because it can create artificial ordering between categories. For example, mapping `No = 0`, `Yes = 1`, and `No internet service = 2` could make the model interpret one category as greater than another, even though these are only labels.

One-hot encoding represents each category as a separate binary feature, preventing the model from assuming a false ranking.

## Numerical Scaling Strategy

Numerical features are scaled using `StandardScaler`.

`StandardScaler` transforms each numerical feature by subtracting the training-set mean and dividing by the training-set standard deviation. This puts numerical features on a comparable scale, which is important for models such as Logistic Regression that are sensitive to feature magnitude.

The scaler must be fitted only on the training data to avoid data leakage.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("binary", "passthrough", binary_features)
    ]
)

`handle_unknown="ignore"` is used in `OneHotEncoder` to prevent errors when unseen categories appear in the test set or future production data. This makes the preprocessing pipeline more robust.

In [11]:
# Fit the preprocessor only on the training data
preprocessor.fit(X_train)
# Transform both training and test sets
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

X_train_processed shape: (5634, 45)
X_test_processed shape: (1409, 45)


After preprocessing, the number of columns increased from 19 to 45 because one-hot encoding expands categorical variables into multiple binary columns.

The number of rows stayed the same, which confirms that preprocessing transformed the feature representation without dropping any observations.

It is important that `X_train_processed` and `X_test_processed` have the same number of columns because the model expects the same feature structure during training and evaluation.

The model learns patterns from 45 processed input features in the training set. The test set must have the same 45 processed features in the same order so predictions are valid.

`handle_unknown="ignore"` ensures that unseen categories in the test set do not break the preprocessing pipeline or create a different feature structure.

The encoder learns the category columns from the training data only. During transformation, both the training and test sets are converted into the same 45-column structure, which is required for valid model training and evaluation.

## Saving the Preprocessor

The fitted preprocessor is saved so it can be reused during modeling and future predictions.

This ensures that the same preprocessing rules learned from the training data are applied consistently. Saving the fitted object also prevents accidental refitting on the wrong data, which could introduce data leakage or create a different feature structure.

In [12]:
import joblib
import os

# Create models directory if it does not exist
os.makedirs("../models", exist_ok=True)

# Save fitted preprocessor
joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']

## Saving Processed Data

The processed training and test datasets are saved for use in the modeling notebook.

This ensures that all models are trained and evaluated on the same preprocessing output and the same train/test split, improving reproducibility.

In [14]:
import numpy as np

# Save processed feature matrices and target arrays
np.save("../models/X_train_processed.npy", X_train_processed)
np.save("../models/X_test_processed.npy", X_test_processed)
np.save("../models/y_train.npy", y_train)
np.save("../models/y_test.npy", y_test)

## Preprocessing Summary

The preprocessing workflow is complete.

Completed steps:
- Dropped the non-predictive `customerID` column.
- Converted `TotalCharges` from string to numeric.
- Imputed hidden missing `TotalCharges` values with 0 based on business logic.
- Encoded the target variable `Churn` as 0/1.
- Separated features and target.
- Created a stratified train/test split.
- Scaled numerical features using `StandardScaler`.
- Encoded categorical features using `OneHotEncoder`.
- Passed through the already-encoded binary feature `SeniorCitizen`.
- Saved the fitted preprocessor and processed train/test datasets for modeling.

The processed feature matrices contain 45 columns after one-hot encoding and are ready for model training.